# Session 2 · Part 2 — Load verified DGAT predictions

**Independent checkpoint:** validate and copy the committed official DGAT prediction table, preserving provenance. This part does not need Session 2, Part 1 to have run.


In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
for candidate in (current, *current.parents):
    if (candidate / "src" / "dgat_tutorial").is_dir():
        tutorial_root = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter inside the hands-on_tutorial directory.")

sys.path.insert(0, str(tutorial_root / "src"))

from dgat_tutorial.checkpoints import tutorial_paths, write_checkpoint

paths = tutorial_paths(tutorial_root)
print(f"Tutorial root: {paths.root}")


In [ ]:
from dgat_tutorial.dgat import load_prediction_metadata, load_prediction_table, write_prediction_artifact

source_path = paths.raw_data / "dgat_predictions.csv"
if not source_path.is_file():
    raise FileNotFoundError(f"Missing verified prediction table: {source_path}")

predicted_proteins = load_prediction_table(str(source_path))
metadata = load_prediction_metadata(source_path)
if metadata is None:
    raise FileNotFoundError(f"Missing provenance sidecar for {source_path}")

print(f"Method: {metadata['method']}")
print(f"Evaluation note: {metadata['evaluation_note']}")
predicted_proteins.head()


In [ ]:
output_path = paths.processed_data / "predicted_proteins.csv"
metadata_path = write_prediction_artifact(
    predicted_proteins,
    output_path,
    method=str(metadata["method"]),
    source=str(metadata["source"]),
    evaluation_note=str(metadata["evaluation_note"]),
)
manifest = write_checkpoint(
    "2.2", [output_path, metadata_path],
    summary={"spots": len(predicted_proteins), "proteins": predicted_proteins.shape[1]},
    start=paths.root,
)
print(f"Checkpoint written: {manifest}")


## Checkpoint

The processed prediction matrix and metadata sidecar can now be used by any later part.